# 43 — Cross-model both-improved on the validated pool (Gemma vs Llama, strong guides)

The original cross-model showcase (25/26/29) gave every model the legacy
27-concept set — never steerability-validated. It found Gemma composes
near-universally (both-improved 97–99%) while Llama struggles (51–59%).
But that handed Llama weak guides; notebook 28 later showed validated
guides lift Llama's joint gains ~40×. This notebook re-asks the family
question with **strong guides**: does the Gemma–Llama gap persist when
both families steer with the validated bf16 pool (notebook 32)?

**Design.** 30 ρ-spanning pairs drawn from pool 32's 40 concepts; three
methods (blend / vote / schedule); k=4, steps=24, the 10 showcase
prompts. Models run fresh here: `meta-llama/Llama-3.1-8B-Instruct` (bf16) and
its AWQ-INT4 variant, `google/gemma-2-2b-it`, `google/gemma-2-9b-it`
(Gemmas bf16). The AWQ column separates the pool effect from
quantization against the legacy-pool table of notebooks 26/29. Headline =
**both-improved rate** (share of runs with every concept's Δ > 0), the
scale-free cross-model statistic, in both expression and angle tracks.

**Caveats (stated, not hidden).** The pool is *Llama-validated*: its
concepts were selected by bf16-Llama steerability, so this is the
stronger test of the family finding (Gemma winning on Llama-favorable
concepts) rather than a symmetric one. Concepts are rebuilt per
tokenizer; Llama uses min_lemmas=11 (8 languages), Gemma English-only
min_lemmas=4 — concepts that do not survive a model's tokenizer are
reported and their pairs dropped for that model, so effective pair counts
differ. ρ is recomputed per model; pairs are selected once, on the pool's
Llama ρ, and shared across models.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.representations import FrameUnembeddingRepresentation
from frames.representations.concept import Concept
from frames.representations.schedulers import RoundRobin
from frames.utils.memory import gc_cuda

In [ ]:
MAX_TOKENS = 3
K = 4
STEPS = 24
N_PAIRS = 30
MAX_USES_PER_CONCEPT = 3

# Llama draws on 8 WordNet languages (min_lemmas=11); Gemma is English-only
# at the paper's model-comparison threshold (min_lemmas=4), exactly as 25/26.
MIN_LEMMAS_LLAMA = 11
MIN_LEMMAS_ENGLISH = 4

POOL = json.loads(Path("resources/32_concept_pool_bf16.json").read_text())
POOL_NAMES = [r["synset"] for r in POOL["pool"]]


def pool_rho(a, b):
    key = f"{a}|{b}" if f"{a}|{b}" in POOL["pairwise_rho"] else f"{b}|{a}"
    return POOL["pairwise_rho"][key]


MODELS = {
    "llama-3.1-8b-bf16": {
        "min_lemmas": MIN_LEMMAS_LLAMA,
        "kwargs": {
            "id": "meta-llama/Llama-3.1-8B-Instruct",
            "torch_dtype": torch.bfloat16,
            "device_map": "auto",
        },
    },
    "llama-3.1-8b-awq": {
        "min_lemmas": MIN_LEMMAS_LLAMA,
        "kwargs": {
            "id": "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
            "device_map": "cuda:0",
        },
    },
    "gemma-2-2b": {
        "min_lemmas": MIN_LEMMAS_ENGLISH,
        "kwargs": {
            "id": "google/gemma-2-2b-it",
            "torch_dtype": torch.bfloat16,
            "device_map": "auto",
        },
    },
    "gemma-2-9b": {
        "min_lemmas": MIN_LEMMAS_ENGLISH,
        "kwargs": {
            "id": "google/gemma-2-9b-it",
            "torch_dtype": torch.bfloat16,
            "device_map": "auto",
        },
    },
}


def chat_fn(model_key):
    if "llama" in model_key:
        return lambda u, a="": (
            f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>{a}"
        )
    if "gemma" in model_key:
        return lambda u, a="": (
            f"<start_of_turn>user\n{u}<end_of_turn>\n<start_of_turn>model\n{a}"
        )
    raise ValueError(model_key)


QUESTIONS = [
    ("What men can be?", "1."),
    ("What women can be?", "1."),
    ("Tell me a short story.", "Once"),
    ("Describe your favorite place.", "My favorite place is"),
    ("What do you like to do on weekends?", "I like"),
    ("Give me some advice for a good life.", "1."),
    ("What is the most important thing in the world?", "The most important thing is"),
    ("Describe the weather today.", "Today the weather is"),
    ("What should I cook for dinner?", "You should"),
    ("Name things that make people happy.", "1."),
]

LOG_PATH = Path("resources/43_validated_pool.jsonl")
PAIRS_PATH = Path("resources/43_pairs.json")
LOG_PATH.unlink(missing_ok=True)

In [ ]:
# 30 pairs evenly spaced along the pool's (Llama) rho spectrum, each concept
# used at most 3 times
all_pairs = sorted(
    (
        {"a": a, "b": b, "rho": pool_rho(a, b)}
        for i, a in enumerate(POOL_NAMES)
        for b in POOL_NAMES[i + 1 :]
    ),
    key=lambda p: p["rho"],
)
targets = [
    all_pairs[round(i * (len(all_pairs) - 1) / (N_PAIRS - 1))]["rho"]
    for i in range(N_PAIRS)
]
PAIRS, uses = [], {}
for t in targets:
    for p in sorted(all_pairs, key=lambda p: abs(p["rho"] - t)):
        key = (p["a"], p["b"])
        if key in {(q["a"], q["b"]) for q in PAIRS}:
            continue
        if uses.get(p["a"], 0) >= MAX_USES_PER_CONCEPT:
            continue
        if uses.get(p["b"], 0) >= MAX_USES_PER_CONCEPT:
            continue
        PAIRS.append(p)
        uses[p["a"]] = uses.get(p["a"], 0) + 1
        uses[p["b"]] = uses.get(p["b"], 0) + 1
        break
if len(PAIRS) < N_PAIRS:
    raise RuntimeError("usage cap prevented an even pair spread")
SYNSETS = sorted({s for p in PAIRS for s in (p["a"], p["b"])})
PAIRS_PATH.write_text(json.dumps({"pairs": PAIRS}, indent=1))
print(
    f"{len(PAIRS)} pairs (rho {PAIRS[0]['rho']:.2f}..{PAIRS[-1]['rho']:.2f}), "
    f"{len(SYNSETS)} concepts -> {PAIRS_PATH}"
)

In [ ]:
def run_model(model_key, spec):
    min_lemmas = spec["min_lemmas"]
    fur = FrameUnembeddingRepresentation.from_model_id(**spec["kwargs"])
    chat = chat_fn(model_key)
    prompts = [chat(u, a) for u, a in QUESTIONS]
    harness = EvaluationHarness(
        fur, LOG_PATH, min_lemmas_per_synset=min_lemmas, max_token_count=MAX_TOKENS
    )
    baseline_texts = harness.generate_baseline(prompts, max_new_tokens=STEPS + 2)

    concept, missing = {}, []
    for name in SYNSETS:
        try:
            c = fur.get_concept_cached(name, min_lemmas, MAX_TOKENS)
            concept[name] = Concept(
                synset=c.synset, tensor=c.tensor.to(fur.model.device)
            )
        except Exception as error:  # noqa: BLE001 - survival is data-dependent
            missing.append(name)
            print(f"  DROPPED {name}: {type(error).__name__}")
    pairs = [p for p in PAIRS if p["a"] not in missing and p["b"] not in missing]
    print(f"[{model_key}] {len(concept)}/{len(SYNSETS)} concepts, {len(pairs)} pairs")
    if not pairs:
        raise RuntimeError(f"[{model_key}] no pairs survived")

    for p in pairs:
        a, b = p["a"], p["b"]
        ca, cb = concept[a], concept[b]
        rho_model = float(ca.rho(cb))
        methods = {
            "blend": lambda: fur.generate_with_topk_guide(
                prompts, guide=Concept.average([ca, cb]), k=K, steps=STEPS
            ),
            "vote": lambda: fur.generate_with_topk_multi_guide(
                prompts, guides=[ca, cb], k=K, steps=STEPS, normalize="zscore"
            ),
            "schedule": lambda: fur.generate_with_topk_multi_guide(
                prompts, guides=[ca, cb], k=K, steps=STEPS, scorer=RoundRobin()
            ),
        }
        for method, generate in methods.items():
            torch.cuda.empty_cache()
            texts, probe = generate()
            harness.evaluate(
                prompts,
                texts,
                [a, b],
                config={
                    "model": model_key,
                    "method": method,
                    "a": a,
                    "b": b,
                    "rho": rho_model,
                    "rho_pool": p["rho"],
                },
                probe=probe,
                baseline_texts=baseline_texts,
                concepts={a: ca, b: cb},
                angles=True,
            )
    print(f"[{model_key}] done")


for model_key, spec in MODELS.items():
    with gc_cuda():
        run_model(model_key, spec)

In [ ]:
parsed = [json.loads(line) for line in LOG_PATH.read_text().strip().split("\n")]
assert {r["config"]["model"] for r in parsed} == set(MODELS), {
    r["config"]["model"] for r in parsed
}

rows = []
for r in parsed:
    c = r["config"]
    da = r["expression"][c["a"]]["delta"]
    db = r["expression"][c["b"]]["delta"]
    ca = r["angles"][c["a"]]["delta"]["token_cos"]
    cb = r["angles"][c["b"]]["delta"]["token_cos"]
    rows.append(
        {
            "model": c["model"],
            "method": c["method"],
            "joint": min(da, db),
            "joint_cos": min(ca, cb),
            "both_improved": min(da, db) > 0,
            "both_improved_cos": min(ca, cb) > 0,
            "ppl_ratio": r["ppl_ratio"],
        }
    )
results = pd.DataFrame(rows)

summary = (
    results.groupby(["model", "method"])
    .agg(
        joint=("joint", "mean"),
        joint_cos=("joint_cos", "mean"),
        both_improved=("both_improved", "mean"),
        both_improved_cos=("both_improved_cos", "mean"),
        ppl_ratio=("ppl_ratio", "mean"),
        n=("joint", "size"),
    )
    .reset_index()
)
summary.to_csv("resources/43_validated_pool_summary.csv", index=False)
summary.round(4)

In [ ]:
ALL_MODELS = ["llama-3.1-8b-awq", "llama-3.1-8b-bf16", "gemma-2-2b", "gemma-2-9b"]
order = ["blend", "vote", "schedule"]
colors = {"blend": "tab:blue", "vote": "tab:orange", "schedule": "tab:green"}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, (metric, label) in zip(
    axes,
    (
        ("both_improved", "both-improved rate (expression)"),
        ("both_improved_cos", "both-improved rate (cosine)"),
    ),
):
    for j, method in enumerate(order):
        vals = [
            summary[(summary["model"] == m) & (summary["method"] == method)][
                metric
            ].iloc[0]
            for m in ALL_MODELS
        ]
        ax.bar(
            [i + (j - 1) * 0.25 for i in range(len(ALL_MODELS))],
            vals,
            width=0.24,
            color=colors[method],
            label=method,
        )
    ax.set_xticks(range(len(ALL_MODELS)), ALL_MODELS, fontsize=8)
    ax.set_ylabel(label)
    ax.axhline(0.5, color="gray", ls="--", lw=0.8)
    ax.set_ylim(0, 1)
axes[0].legend(fontsize=8)
plt.suptitle("Both-improved across families on the validated (Llama) pool")
plt.tight_layout()
plt.savefig("resources/43_validated_pool.png", dpi=150)
plt.show()

print("both-improved (expression / cosine) by model, best method:")
for m in ALL_MODELS:
    sub = summary[summary["model"] == m].sort_values("both_improved", ascending=False)
    top = sub.iloc[0]
    print(
        f"  {m:18s} {top['method']:9s} "
        f"{top['both_improved']:.0%} / {top['both_improved_cos']:.0%}"
    )

print(
    "\nlegacy-pool reference (notebooks 26/29): "
    "llama ~52-59%, gemma-2b 99.7%, gemma-9b 97-98%"
)